# Preparation for USE

# Einwohnerdaten

In [1]:
import pandas as pd
import geopandas as gpd

PATH_RAW = "../data/prep/"

In [2]:
df_profile = pd.read_excel("../data/prep/ueberarbeitet_Stadtteilprofile Barmbek_Sued-Berichtsjahre-2013-2023.xlsx", header=[0,1])

In [3]:
df_profile.columns = [
    "_".join([str(c) for c in col if c]) 
    for col in df_profile.columns
]

In [4]:
df_profile = df_profile.dropna(how="all")
df_profile = df_profile.reset_index(drop=True)

In [5]:
df_profile.to_csv("../data/mart/df_profile_mart.csv", index=False)

In [6]:
df_profile.head()

,Berichtsjahr_Jahr,Bevölkerung und Haushalte_Anzahl der Einwohner: innen,Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren,Bevölkerung und Haushalte_Anteil Kinder und Jugendlicher unter 18 Jahren an der Gesamt-bevölkerung,Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren,Bevölkerung und Haushalte_Anteil älterer Einwohner: innen über 64 Jahren an der Gesamt-bevölkerung,Bevölkerung und Haushalte_Anzahl ausländischer Einwohner: innen,Bevölkerung und Haushalte_Anteil ausländischer Einwohner: innen an der Gesamt-bevölkerung,Bevölkerung und Haushalte_Anzahl der Einwohner: innen mit Migrations-hintergrund,Bevölkerung und Haushalte_Anteil der Einwohner: innen mit Migrations-hintergrund an der Gesamt-bevölkerung,...,Infrastruktur und Verkehr_Anzahl der Schüler:innen der Sekundarstufe I (nach Wohnort),Infrastruktur und Verkehr_Anteil der Schüler:innen in Stadtteil-schulen an allen Schüler:innen der Sekundarstufe I (nach Wohnort),Infrastruktur und Verkehr_Anteil der Schüler:innen in Gymnasien an allen Schüler:innen der Sekundarstufe I (nach Wohnort),Infrastruktur und Verkehr_Anzahl der nieder-gelassenen Ärztinnen/Ärzte,Infrastruktur und Verkehr_Anzahl der Allgemein-ärztinnen/-ärzte,Infrastruktur und Verkehr_Anzahl der Zahn-ärztinnen/-ärzte,Infrastruktur und Verkehr_Anzahl der Apotheken,Infrastruktur und Verkehr_Anzahl privater PKW,Infrastruktur und Verkehr_Anzahl der privaten PKW je 1 000 Einwohner: innen,Infrastruktur und Verkehr_Anzahl Elektro-PKW
0,2024,37091,4096,11.0,5057,13.6,6819,18.4,12704,34.3,...,1107,51.6,45.9,65,21,29,6,9746,263.0,587.0
1,2023,36705,4078,11.1,5004,13.6,6603,18.0,12315,33.6,...,1107,52.1,45.1,61,21,27,6,9885,269.0,812.0
2,2022,36452,4077,11.2,5020,13.8,6264,17.2,11809,32.4,...,1057,50.1,47.2,54,15,25,6,10073,276.0,675.0
3,2021,35709,3886,10.9,4976,13.9,5261,14.7,10609,29.7,...,978,50.6,46.5,51,14,25,6,10332,289.0,333.0
4,2020,35880,3906,10.9,5012,14.0,5096,14.2,10331,28.8,...,977,50.7,46.3,52,15,27,6,10259,286.0,251.0


In [7]:
df_profile.columns

Index(['Berichtsjahr_Jahr',
       'Bevölkerung und Haushalte_Anzahl der Einwohner: innen',
       'Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren',
       'Bevölkerung und Haushalte_Anteil Kinder und Jugendlicher unter 18 Jahren an der Gesamt-bevölkerung',
       'Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren',
       'Bevölkerung und Haushalte_Anteil älterer Einwohner: innen über 64 Jahren an der Gesamt-bevölkerung',
       'Bevölkerung und Haushalte_Anzahl ausländischer Einwohner: innen',
       'Bevölkerung und Haushalte_Anteil ausländischer Einwohner: innen an der Gesamt-bevölkerung',
       'Bevölkerung und Haushalte_Anzahl der Einwohner: innen mit Migrations-hintergrund',
       'Bevölkerung und Haushalte_Anteil der Einwohner: innen mit Migrations-hintergrund an der Gesamt-bevölkerung',
       'Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren mit Migrations-hintergrund',
       'Bevölkerung und 

# Untersuchung Haushalte und alles in englisch übersetzt

In [8]:
import pandas as pd

# =========================
# 1. COPY (wichtig!)
# =========================
df_households = df_profile.copy()

# =========================
# 2. RENAME
# =========================
df_households = df_households.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anteil der Haushalte, in denen nur eine Person lebt, an allen Haushalten": "single_pct",
    "Bevölkerung und Haushalte_Anteil der Haushalte, in denen Kinder leben, an allen Haushalten": "children_pct"
})

# =========================
# 3. CLEAN (Komma → Punkt)
# =========================
df_households["single_pct"] = df_households["single_pct"].astype(str).str.replace(",", ".").astype(float)
df_households["children_pct"] = df_households["children_pct"].astype(str).str.replace(",", ".").astype(float)

# =========================
# 4. OTHER
# =========================
df_households["other_pct"] = 100 - df_households["single_pct"] - df_households["children_pct"]

# =========================
# 5. DISTRICT
# =========================
df_households["district"] = "Barmbek-Süd"

# =========================
# 6. LONG FORMAT
# =========================
df_households_long = df_households.melt(
    id_vars=["year", "district"],
    value_vars=["single_pct", "children_pct", "other_pct"],
    var_name="group",
    value_name="value"
)

# =========================
# 7. LABELS
# =========================
df_households_long["group"] = df_households_long["group"].replace({
    "single_pct": "Single households",
    "children_pct": "Households with children",
    "other_pct": "Other households"
})

# =========================
# 8. EXPORT
# =========================
df_households_long.to_csv(
    "../data/mart/mart_households_structure.csv",
    index=False
)

# =========================
# 9. CHECK
# =========================
print(df_households_long.head())

   year     district              group  value
0  2024  Barmbek-Süd  Single households   68.9
1  2023  Barmbek-Süd  Single households   68.6
2  2022  Barmbek-Süd  Single households   68.2
3  2021  Barmbek-Süd  Single households   68.6
4  2020  Barmbek-Süd  Single households   68.6


# Infrastructure Barmbek Süd

In [9]:
# =========================
# INFRASTRUCTURE (COUNTS)
# =========================

df_inf = df_profile.copy()

# Rename (JETZT RICHTIG)
df_inf = df_inf.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Infrastruktur und Verkehr_Kindergärten und Vorschul-klassen für Kinder im Alter von 3 Jahren bis Schuleintritt": "kitas",
    "Infrastruktur und Verkehr_Anzahl der Grundschulen": "primary_schools",
    "Infrastruktur und Verkehr_Anzahl der nieder-gelassenen Ärztinnen/Ärzte": "doctors",
    "Infrastruktur und Verkehr_Anzahl der Allgemein-ärztinnen/-ärzte": "general_doctors",
    "Infrastruktur und Verkehr_Anzahl der Zahn-ärztinnen/-ärzte": "dentists",
    "Infrastruktur und Verkehr_Anzahl der Apotheken": "pharmacies"
})

# District
df_inf["district"] = "Barmbek-Süd"

# Auswahl
df_inf = df_inf[[
    "year", "district",
    "kitas",
    "primary_schools",
    "doctors", "general_doctors",
    "dentists", "pharmacies"
]]

# Long format
df_inf_counts = df_inf.melt(
    id_vars=["year", "district"],
    var_name="indicator",
    value_name="value"
)

# Export
df_inf_counts.to_csv(
    "../data/mart/mart_infrastructure_counts_barmbek.csv",
    index=False
)

print(df_inf_counts.head())

   year     district indicator  value
0  2024  Barmbek-Süd     kitas     27
1  2023  Barmbek-Süd     kitas     28
2  2022  Barmbek-Süd     kitas     29
3  2021  Barmbek-Süd     kitas     27
4  2020  Barmbek-Süd     kitas     27


# Education Barmbek Süd

In [10]:
# =========================
# EDUCATION (%)
# =========================

df_edu = df_profile.copy()

# Rename
df_edu = df_edu.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Infrastruktur und Verkehr_Anteil der Schüler:innen in Stadtteil-schulen an allen Schüler:innen der Sekundarstufe I (nach Wohnort)": "district_school_pct",
    "Infrastruktur und Verkehr_Anteil der Schüler:innen in Gymnasien an allen Schüler:innen der Sekundarstufe I (nach Wohnort)": "gymnasium_pct"
})

# Clean (falls nötig)
df_edu["district_school_pct"] = df_edu["district_school_pct"].astype(float)
df_edu["gymnasium_pct"] = df_edu["gymnasium_pct"].astype(float)

# District
df_edu["district"] = "Barmbek-Süd"

# Auswahl (nur %!)
df_edu = df_edu[[
    "year", "district",
    "district_school_pct", "gymnasium_pct"
]]

# Long format
df_edu_pct = df_edu.melt(
    id_vars=["year", "district"],
    var_name="indicator",
    value_name="value"
)

# Export
df_edu_pct.to_csv(
    "../data/mart/mart_education_pct_barmbek.csv",
    index=False
)

print(df_edu_pct.head())

   year     district            indicator  value
0  2024  Barmbek-Süd  district_school_pct   51.6
1  2023  Barmbek-Süd  district_school_pct   52.1
2  2022  Barmbek-Süd  district_school_pct   50.1
3  2021  Barmbek-Süd  district_school_pct   50.6
4  2020  Barmbek-Süd  district_school_pct   50.7


# Unterschiedliche Altersstrukturen

In [11]:
import pandas as pd

# =========================
# 1. COPY
# =========================
df_age = df_profile.copy()

# =========================
# 2. RENAME
# =========================
df_age = df_age.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anteil Kinder und Jugendlicher unter 18 Jahren an der Gesamt-bevölkerung": "children_pct",
    "Bevölkerung und Haushalte_Anteil älterer Einwohner: innen über 64 Jahren an der Gesamt-bevölkerung": "elderly_pct"
})

# =========================
# 3. CLEAN
# =========================
df_age["children_pct"] = df_age["children_pct"].astype(str).str.replace(",", ".").astype(float)
df_age["elderly_pct"] = df_age["elderly_pct"].astype(str).str.replace(",", ".").astype(float)

# =========================
# 4. OTHER (wichtig!)
# =========================
df_age["other_pct"] = 100 - df_age["children_pct"] - df_age["elderly_pct"]

# =========================
# 5. DISTRICT
# =========================
df_age["district"] = "Barmbek-Süd"

# =========================
# 6. LONG FORMAT
# =========================
df_age_long = df_age.melt(
    id_vars=["year", "district"],
    value_vars=["children_pct", "elderly_pct", "other_pct"],
    var_name="group",
    value_name="value"
)

# =========================
# 7. LABELS
# =========================
df_age_long["group"] = df_age_long["group"].replace({
    "children_pct": "Children (<18)",
    "elderly_pct": "Elderly (65+)",
    "other_pct": "Working-age population"
})

# =========================
# 8. EXPORT
# =========================
df_age_long.to_csv(
    "../data/mart/mart_age_structure.csv",
    index=False
)

# =========================
# 9. CHECK
# =========================
print(df_age_long.head())

   year     district           group  value
0  2024  Barmbek-Süd  Children (<18)   11.0
1  2023  Barmbek-Süd  Children (<18)   11.1
2  2022  Barmbek-Süd  Children (<18)   11.2
3  2021  Barmbek-Süd  Children (<18)   10.9
4  2020  Barmbek-Süd  Children (<18)   10.9


# Autobesitzer

In [12]:
import pandas as pd

# =========================
# 1. COPY
# =========================
df_cars = df_profile.copy()

# =========================
# 2. RENAME
# =========================
df_cars = df_cars.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Infrastruktur und Verkehr_Anzahl der privaten PKW je 1 000 Einwohner: innen": "cars_per_1000"
})

# =========================
# 3. CLEAN
# =========================
df_cars["cars_per_1000"] = df_cars["cars_per_1000"].astype(str).str.replace(",", ".").astype(float)

# =========================
# 4. INTO %
# =========================
df_cars["car_owner_pct"] = df_cars["cars_per_1000"] / 10
df_cars["no_car_pct"] = 100 - df_cars["car_owner_pct"]

# =========================
# 5. DISTRICT
# =========================
df_cars["district"] = "Barmbek-Süd"

# =========================
# 6. LONG FORMAT
# =========================
df_cars_long = df_cars.melt(
    id_vars=["year", "district"],
    value_vars=["car_owner_pct", "no_car_pct"],
    var_name="group",
    value_name="value"
)

# =========================
# 7. LABELS
# =========================
df_cars_long["group"] = df_cars_long["group"].replace({
    "car_owner_pct": "Car owners",
    "no_car_pct": "No car"
})

# =========================
# 8. EXPORT
# =========================
df_cars_long.to_csv(
    "../data/mart/mart_car_ownership.csv",
    index=False
)

print(df_cars_long.head())

   year     district       group  value
0  2024  Barmbek-Süd  Car owners   26.3
1  2023  Barmbek-Süd  Car owners   26.9
2  2022  Barmbek-Süd  Car owners   27.6
3  2021  Barmbek-Süd  Car owners   28.9
4  2020  Barmbek-Süd  Car owners   28.6


# Migraten im Stadtteil

In [13]:
import pandas as pd

# =========================
# 1. COPY
# =========================
df_migration = df_profile.copy()

# =========================
# 2. RENAME
# =========================
df_migration = df_migration.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anteil der Einwohner: innen mit Migrations-hintergrund an der Gesamt-bevölkerung": "migration_pct"
})

# =========================
# 3. CLEAN
# =========================
df_migration["migration_pct"] = (
    df_migration["migration_pct"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)

# =========================
# 4. NON-MIGRATION
# =========================
df_migration["no_migration_pct"] = 100 - df_migration["migration_pct"]

# =========================
# 5. DISTRICT
# =========================
df_migration["district"] = "Barmbek-Süd"

# =========================
# 6. LONG FORMAT
# =========================
df_migration_long = df_migration.melt(
    id_vars=["year", "district"],
    value_vars=["migration_pct", "no_migration_pct"],
    var_name="group",
    value_name="value"
)

# =========================
# 7. LABELS
# =========================
df_migration_long["group"] = df_migration_long["group"].replace({
    "migration_pct": "With migration background",
    "no_migration_pct": "Without migration background"
})

# =========================
# 8. EXPORT
# =========================
df_migration_long.to_csv(
    "../data/mart/mart_migration_structure.csv",
    index=False
)

print(df_migration_long.head())

   year     district                      group  value
0  2024  Barmbek-Süd  With migration background   34.3
1  2023  Barmbek-Süd  With migration background   33.6
2  2022  Barmbek-Süd  With migration background   32.4
3  2021  Barmbek-Süd  With migration background   29.7
4  2020  Barmbek-Süd  With migration background   28.8


# Nur 1 Berichtsjahr

In [14]:
df_profile_small = df_profile[[
    "Berichtsjahr_Jahr",
    "Bevölkerung und Haushalte_Anzahl der Einwohner: innen",
    "Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren",
    "Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren"
]]

In [15]:
print(df_profile.head())
print(df_profile.columns)
print(df_profile.info())

   Berichtsjahr_Jahr  Bevölkerung und Haushalte_Anzahl der Einwohner: innen  \
0               2024                                              37091       
1               2023                                              36705       
2               2022                                              36452       
3               2021                                              35709       
4               2020                                              35880       

   Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren  \
0                                               4096                              
1                                               4078                              
2                                               4077                              
3                                               3886                              
4                                               3906                              

   Bevölkerung und Haushal

In [16]:
print(df_profile_small.head())
print(df_profile_small.columns)
print(df_profile_small.info())

   Berichtsjahr_Jahr  Bevölkerung und Haushalte_Anzahl der Einwohner: innen  \
0               2024                                              37091       
1               2023                                              36705       
2               2022                                              36452       
3               2021                                              35709       
4               2020                                              35880       

   Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren  \
0                                               4096                              
1                                               4078                              
2                                               4077                              
3                                               3886                              
4                                               3906                              

   Bevölkerung und Haushal

In [17]:
df_profile_small["Berichtsjahr_Jahr"] = pd.to_numeric(
    df_profile_small["Berichtsjahr_Jahr"], errors="coerce"
)

In [18]:
df_profile_latest = df_profile_small[df_profile_small["Berichtsjahr_Jahr"] == 2024]

# Für sociodemografische Eckdaten über Stadtteil

In [19]:
df_pop = df_profile.copy()

df_pop = df_pop.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anzahl der Einwohner: innen": "population"
})

df_pop_trend = df_pop[["year", "population"]]

df_pop_trend.to_csv("../data/mart/mart_population_trend.csv", index=False)

In [20]:
df_pop = df_profile.copy()

df_pop = df_pop.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anzahl der Einwohner: innen": "population",
    "Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren": "children",
    "Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren": "elderly"
})

# district hinzufügen
df_pop["district"] = "Barmbek-Süd"

# Long Format
df_long = df_pop.melt(
    id_vars=["year", "district"],
    value_vars=["population", "children", "elderly"],
    var_name="group",
    value_name="value"
)

df_long["group"] = df_long["group"].replace({
    "population": "Total Population",
    "children": "Children (<18)",
    "elderly": "Elderly (65+)"
})

# Export
df_long.to_csv("../data/mart/mart_population_long.csv", index=False)

In [21]:
df_long.head()


,year,district,group,value
0,2024,Barmbek-Süd,Total Population,37091
1,2023,Barmbek-Süd,Total Population,36705
2,2022,Barmbek-Süd,Total Population,36452
3,2021,Barmbek-Süd,Total Population,35709
4,2020,Barmbek-Süd,Total Population,35880


# Zu Einwohnerzahl Spalte district hinzufügen, damit Verbindung zu Geodaten

In [22]:
df_profile_latest["district"] = "Barmbek-Süd"

# Merge of geadata and socio data

# Again load all prep geojsons

In [23]:

# Transport & Infrastruktur
gdf_stops = gpd.read_file(PATH_RAW + "prep_public_transport_barmbek.geojson")
gdf_crossings = gpd.read_file(PATH_RAW + "prep_crossings_barmbek.geojson")
gdf_bike = gpd.read_file(PATH_RAW + "prep_bikelane_barmbek.geojson")

gdf_bike_proj = gdf_bike.to_crs(epsg=25832)

bike_length_km = gdf_bike_proj.length.sum() / 1000

# Grün & Umwelt
gdf_green = gpd.read_file(PATH_RAW + "prep_green_barmbek.geojson")
gdf_park = gpd.read_file(PATH_RAW + "prep_park_barmbek.geojson")
gdf_greenplan = gpd.read_file(PATH_RAW + "prep_greenplan_barmbek.geojson")

# Mobilität
gdf_stadtrad = gpd.read_file(PATH_RAW + "prep_stadtrad_barmbek.geojson")
gdf_emobility = gpd.read_file(PATH_RAW + "prep_emobility_barmbek.geojson")

# Verkehr & Straßen
gdf_streets = gpd.read_file(PATH_RAW + "prep_streets_cover_barmbek.geojson")
gdf_highway = gpd.read_file(PATH_RAW + "prep_highway_barmbek.geojson")
gdf_crosswalks = gpd.read_file(PATH_RAW + "prep_fussgaengerueberwege_barmbek.geojson")

# Sonstiges
gdf_switch = gpd.read_file(PATH_RAW + "prep_switch_barmbek.geojson")
gdf_lsa = gpd.read_file(PATH_RAW + "prep_lsa_barmbek.geojson")

In [24]:
pt_stops = len(gdf_stops)
crossings = len(gdf_crossings)
stadtrad = len(gdf_stadtrad)
emobility = len(gdf_emobility)
lsa = len(gdf_lsa)
street_segments = len(gdf_streets)

green_area = float(gdf_greenplan["FLAECHE_QM"].sum())
bike_km = float(bike_length_km)

In [25]:
import pandas as pd

df_geo = pd.DataFrame([{
    "district": "Barmbek-Süd",
    "pt_stops": pt_stops,
    "crossings": crossings,
    "stadtrad": stadtrad,
    "emobility": emobility,
    "lsa": lsa,
    "green_area_m2": green_area,
    "street_segments": street_segments,
    "bike_km": bike_km
}])

## Check of both tables

In [26]:
df_geo.head()


,district,pt_stops,crossings,stadtrad,emobility,lsa,green_area_m2,street_segments,bike_km
0,Barmbek-Süd,19,15,10,20,27,311652.0,520,12.042108


In [27]:
df_profile_latest.head()

,Berichtsjahr_Jahr,Bevölkerung und Haushalte_Anzahl der Einwohner: innen,Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren,Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren,district
0,2024,37091,4096,5057,Barmbek-Süd


# Merge

In [28]:
df_final = df_geo.merge(
    df_profile_latest,
    on="district",
    how="left"
)

In [29]:
df_final.head()

,district,pt_stops,crossings,stadtrad,emobility,lsa,green_area_m2,street_segments,bike_km,Berichtsjahr_Jahr,Bevölkerung und Haushalte_Anzahl der Einwohner: innen,Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren,Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren
0,Barmbek-Süd,19,15,10,20,27,311652.0,520,12.042108,2024,37091,4096,5057


In [30]:
df_final.rename(columns={
    "Bevölkerung und Haushalte_Anzahl der Einwohner: innen": "population",
    "Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren": "children",
    "Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren": "elderly"
}, inplace=True)

In [31]:
df_final.to_csv("../data/mart/walkability_data_mart.csv", index=False, sep=";")

# Walkability Score nach Chatjpt 

## Normalize all data, because they are in different scales

In [32]:
df_score = df_final.copy()

df_score["pt_stops_norm"] = df_score["pt_stops"] / df_score["pt_stops"].max()
df_score["crossings_norm"] = df_score["crossings"] / df_score["crossings"].max()
df_score["stadtrad_norm"] = df_score["stadtrad"] / df_score["stadtrad"].max()
df_score["emobility_norm"] = df_score["emobility"] / df_score["emobility"].max()
df_score["green_norm"] = df_score["green_area_m2"] / df_score["green_area_m2"].max()
df_score["bike_norm"] = df_score["bike_km"] / df_score["bike_km"].max()

## Count the score

In [33]:
df_score["walkability_score"] = (
    df_score["pt_stops_norm"] +
    df_score["crossings_norm"] +
    df_score["stadtrad_norm"] +
    df_score["emobility_norm"] +
    df_score["green_norm"] +
    df_score["bike_norm"]
) / 6

## Check

In [34]:
df_score[["district", "walkability_score"]]

,district,walkability_score
0,Barmbek-Süd,1.0


In [35]:
df_final.to_csv("../data/mart/walkability_data_mart.csv", index=False)

In [36]:
df_final.head()

,district,pt_stops,crossings,stadtrad,emobility,lsa,green_area_m2,street_segments,bike_km,Berichtsjahr_Jahr,population,children,elderly
0,Barmbek-Süd,19,15,10,20,27,311652.0,520,12.042108,2024,37091,4096,5057


# Age Structure

In [37]:
df_age = df_profile.copy()

df_age = df_age.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anzahl der Einwohner: innen": "population",
    "Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren": "children",
    "Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren": "elderly"
})

# Nur letztes Jahr nehmen (für klare Aussage)
df_latest = df_age.sort_values("year", ascending=False).head(1)

df_age_structure = pd.DataFrame({
    "group": ["Children (<18)", "Elderly (65+)", "Others"],
    "value": [
        df_latest["children"].values[0],
        df_latest["elderly"].values[0],
        df_latest["population"].values[0]
        - df_latest["children"].values[0]
        - df_latest["elderly"].values[0]
    ]
})

df_age_structure.to_csv("../data/mart/mart_age_structure.csv", index=False)

In [49]:
import pandas as pd

# =========================
# COPY SOURCE DATA
# =========================
df_age = df_profile.copy()

# =========================
# RENAME COLUMNS
# =========================
df_age = df_age.rename(columns={
    "Berichtsjahr_Jahr": "year",
    "Bevölkerung und Haushalte_Anzahl der Einwohner: innen": "population",
    "Bevölkerung und Haushalte_Anzahl der Kinder und Jugendlichen unter 18 Jahren": "children",
    "Bevölkerung und Haushalte_Anzahl älterer Einwohner: innen über 64 Jahren": "elderly"
})

# =========================
# KEEP ONLY RELEVANT COLUMNS
# =========================
df_age = df_age[[
    "year",
    "population",
    "children",
    "elderly"
]].dropna()

# =========================
# BUILD LONG FORMAT (ALL YEARS)
# =========================
rows = []

for _, row in df_age.iterrows():
    
    # Children
    rows.append({
        "year": row["year"],
        "group": "Children (<18)",
        "value": row["children"]
    })

    # Elderly
    rows.append({
        "year": row["year"],
        "group": "Elderly (65+)",
        "value": row["elderly"]
    })

    # Others
    rows.append({
        "year": row["year"],
        "group": "Others",
        "value": row["population"] - row["children"] - row["elderly"]
    })

# =========================
# FINAL DATAFRAME
# =========================
df_age_structure = pd.DataFrame(rows)

# =========================
# SORT (OPTIONAL)
# =========================
df_age_structure = df_age_structure.sort_values(
    by=["year", "group"]
)

# =========================
# EXPORT
# =========================
df_age_structure.to_csv(
    "../data/mart/mart_age_structure.csv",
    index=False
)

# =========================
# CHECK
# =========================
print(df_age_structure.head(15))
print(df_age_structure["year"].unique())

    year           group  value
33  2013  Children (<18)   3105
34  2013   Elderly (65+)   5100
35  2013          Others  24577
30  2014  Children (<18)   3183
31  2014   Elderly (65+)   5092
32  2014          Others  24841
27  2015  Children (<18)   3353
28  2015   Elderly (65+)   5097
29  2015          Others  25231
24  2016  Children (<18)   3625
25  2016   Elderly (65+)   5058
26  2016          Others  26109
21  2017  Children (<18)   3703
22  2017   Elderly (65+)   4992
23  2017          Others  26523
[2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]


# Bike lane

In [38]:
import pandas as pd
import geopandas as gpd

# =========================
# 1. LOAD
# =========================
gdf = gpd.read_file("../data/prep/prep_bikelane_barmbek.geojson")

# =========================
# 2. CLEAN
# =========================
# Länge in km
gdf["length_km"] = gdf["laenge"] / 1000

# =========================
# 3. AGGREGATE
# =========================
df_bike = gdf.groupby("status")["length_km"].sum().reset_index()

# =========================
# 4. ADD CONTEXT
# =========================
df_bike["district"] = "Barmbek-Süd"
df_bike["year"] = 2024

# =========================
# 5. RENAME
# =========================
df_bike = df_bike.rename(columns={
    "status": "category",
    "length_km": "value"
})

# 6. EXPORT
# =========================
df_bike.to_csv(
    "../data/mart/mart_bike_barmbek.csv",
    index=False
)

print(df_bike)

                              category  value     district  year
0                             Existing  6.002  Barmbek-Süd  2024
1        Priority network improvements  6.315  Barmbek-Süd  2024
2  Sections with improvement potential  0.023  Barmbek-Süd  2024
3                   Under construction  3.944  Barmbek-Süd  2024


In [39]:
print(df_bike["category"].unique())

<StringArray>
[                           'Existing',       'Priority network improvements',
 'Sections with improvement potential',                  'Under construction']
Length: 4, dtype: str


# Greenplan 

In [40]:
import pandas as pd
import geopandas as gpd

# LOAD
gdf = gpd.read_file("../data/prep/prep_greenplan_barmbek.geojson")

# AGGREGATE
df_green_summary = (
    gdf.groupby("green_type")["FLAECHE_QM"]
    .sum()
    .reset_index()
)

# KM²
df_green_summary["value"] = df_green_summary["FLAECHE_QM"] / 1_000_000

df_green_summary["value"] = gdf.groupby("green_type")["FLAECHE_QM"].sum().values

# CLEAN
df_green_summary = df_green_summary.rename(columns={
    "green_type": "category"
})

df_green_summary["district"] = "Barmbek-Süd"
df_green_summary["year"] = 2024

df_green_summary["value"] = (
    df_green_summary["FLAECHE_QM"] /
    df_green_summary["FLAECHE_QM"].sum()
)

# EXPORT
df_green_summary.to_csv(
    "../data/mart/mart_green_barmbek.csv",
    index=False
)

print(df_green_summary)

           category  FLAECHE_QM     value     district  year
0  Allotment garden        5135  0.016477  Barmbek-Süd  2024
1         Other use        5924  0.019008  Barmbek-Süd  2024
2              Park      213046  0.683602  Barmbek-Süd  2024
3        Playground       87547  0.280913  Barmbek-Süd  2024


# Streets cover mart

In [41]:
import geopandas as gpd
import pandas as pd

# =========================
# LOAD
# =========================
streets = gpd.read_file("../data/prep/prep_streets_cover_barmbek.geojson")

# =========================
# SELECT
# =========================
streets = streets[[
    "wegeart",
    "fahrstreifenanzahl_in_beide_richtungen",
    "geschwindigkeit",
    "strassendeckschicht",
    "geometry"
]].dropna()

# =========================
# RENAME
# =========================
streets = streets.rename(columns={
    "wegeart": "road_type",
    "fahrstreifenanzahl_in_beide_richtungen": "lanes",
    "geschwindigkeit": "speed_raw",
    "strassendeckschicht": "surface_type"
})

# =========================
# SPEED CLEAN
# =========================
streets["speed_kmh"] = (
    streets["speed_raw"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)

# =========================
# SURFACE CLEAN
# =========================
streets["surface_type_clean"] = streets["surface_type"].astype(str)

streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Asphaltbetone.*", "Asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Offenporiger Asphalt.*", "Porous asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Splittmastixasphalt.*", "Stone mastic asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Gussasphalt.*", "Mastic asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Pflaster.*", "Paving stones", regex=True)

# =========================
# WALKABILITY SCORE (NEU & KLAR)
# =========================
def calc_walkability(row):
    score = 0

    # SPEED (stark gewichtet)
    if row["speed_kmh"] <= 30:
        score += 3
    elif row["speed_kmh"] <= 40:
        score += 1
    else:
        score -= 2

    # LANES (breite Straßen schlechter)
    if row["lanes"] <= 2:
        score += 2
    elif row["lanes"] == 3:
        score += 0
    else:
        score -= 2

    # SURFACE (entscheidend)
    if row["surface_type_clean"] in ["Asphalt", "Porous asphalt"]:
        score += 2
    elif row["surface_type_clean"] == "Stone mastic asphalt":
        score += 1
    elif row["surface_type_clean"] == "Mastic asphalt":
        score += 0
    elif row["surface_type_clean"] == "Paving stones":
        score -= 3   # ❗ bewusst stark negativ
    else:
        score -= 1

    return score

streets["walkability_score"] = streets.apply(calc_walkability, axis=1)

# =========================
# CLASSIFICATION (NUR EINMAL!)
# =========================
def classify_walkability(score):
    if score >= 4:
        return "High walkability"
    elif score >= 1:
        return "Medium walkability"
    else:
        return "Low walkability"

streets["walkability_class"] = streets["walkability_score"].apply(classify_walkability)

# =========================
# CONTEXT
# =========================
streets["district"] = "Barmbek-Süd"
streets["year"] = 2024

# =========================
# FINAL
# =========================
streets_mart = streets[[
    "geometry",
    "road_type",
    "surface_type_clean",
    "speed_kmh",
    "lanes",
    "walkability_score",
    "walkability_class",
    "district",
    "year"
]]

# =========================
# EXPORT
# =========================
streets_mart.to_file(
    "../data/mart/mart_street_walkability.geojson",
    driver="GeoJSON"
)

# =========================
# CHECK
# =========================
print(streets_mart["walkability_class"].value_counts())

walkability_class
Medium walkability    250
High walkability      239
Low walkability        30
Name: count, dtype: int64


In [42]:
import geopandas as gpd
import pandas as pd

# =========================
# LOAD
# =========================
streets = gpd.read_file("../data/prep/prep_streets_cover_barmbek.geojson")

# =========================
# SELECT
# =========================
streets = streets[[
    "wegeart",
    "fahrstreifenanzahl_in_beide_richtungen",
    "geschwindigkeit",
    "strassendeckschicht",
    "geometry"
]].dropna()

# =========================
# RENAME
# =========================
streets = streets.rename(columns={
    "wegeart": "road_type",
    "fahrstreifenanzahl_in_beide_richtungen": "lanes",
    "geschwindigkeit": "speed_raw",
    "strassendeckschicht": "surface_type"
})

# =========================
# SPEED CLEAN
# =========================
streets["speed_kmh"] = (
    streets["speed_raw"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)

# =========================
# SURFACE CLEAN
# =========================
streets["surface_type_clean"] = streets["surface_type"].astype(str)

streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Asphaltbetone.*", "Asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Offenporiger Asphalt.*", "Porous asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Splittmastixasphalt.*", "Stone mastic asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Gussasphalt.*", "Mastic asphalt", regex=True)
streets["surface_type_clean"] = streets["surface_type_clean"].str.replace(r".*Pflaster.*", "Paving stones", regex=True)

# =========================
# WALKABILITY SCORE (0–7)
# =========================
def calc_walkability(row):
    score = 0

    # SPEED
    if row["speed_kmh"] <= 30:
        score += 3
    elif row["speed_kmh"] <= 40:
        score += 2
    else:
        score += 0

    # LANES
    if row["lanes"] <= 2:
        score += 2
    elif row["lanes"] == 3:
        score += 1
    else:
        score += 0

    # SURFACE
    if row["surface_type_clean"] in ["Asphalt", "Porous asphalt"]:
        score += 2
    elif row["surface_type_clean"] == "Stone mastic asphalt":
        score += 1
    elif row["surface_type_clean"] == "Paving stones":
        score += 0
    else:
        score += 1

    return score

streets["walkability_score"] = streets.apply(calc_walkability, axis=1)

# =========================
# CLASSIFICATION
# =========================
def classify_walkability(score):
    if score >= 5:
        return "High walkability"
    elif score >= 3:
        return "Medium walkability"
    else:
        return "Low walkability"

streets["walkability_class"] = streets["walkability_score"].apply(classify_walkability)

# =========================
# CONTEXT
# =========================
streets["district"] = "Barmbek-Süd"
streets["year"] = 2024

# =========================
# FINAL
# =========================
streets_mart = streets[[
    "geometry",
    "road_type",
    "surface_type_clean",
    "speed_kmh",
    "lanes",
    "walkability_score",
    "walkability_class",
    "district",
    "year"
]]

# =========================
# EXPORT
# =========================
streets_mart.to_file(
    "../data/mart/mart_street_walkability.geojson",
    driver="GeoJSON"
)

# =========================
# CHECK
# =========================
print(streets_mart["walkability_score"].describe())
print(streets_mart["walkability_class"].value_counts())

count    519.000000
mean       5.175337
std        1.765742
min        2.000000
25%        3.000000
50%        5.000000
75%        7.000000
max        7.000000
Name: walkability_score, dtype: float64
walkability_class
High walkability      320
Medium walkability    183
Low walkability        16
Name: count, dtype: int64


# Crossings von walkability score

In [43]:
import geopandas as gpd

# LOAD
gdf_streets = gpd.read_file("../data/mart/mart_street_walkability.geojson")
gdf_crossings = gpd.read_file("../data/prep/prep_crossings_barmbek.geojson")

# CRS angleichen (wichtig!)
gdf_crossings = gdf_crossings.to_crs(gdf_streets.crs)

# Spatial Join (nächste Straße finden)
gdf_crossings_joined = gpd.sjoin_nearest(
    gdf_crossings,
    gdf_streets[["geometry", "walkability_score", "walkability_class"]],
    how="left"
)

/opt/miniconda3/envs/python_neu_nicole_12012026/lib/python3.13/site-packages/geopandas/array.py:407: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [44]:
gdf_crossings_joined.to_file(
    "../data/mart/mart_crossings_walkability.geojson",
    driver="GeoJSON"
)

# Final score streets + infrastructure from streets

In [45]:
import geopandas as gpd
import pandas as pd

# =========================
# LOAD DATA
# =========================
gdf_streets = gpd.read_file("../data/mart/mart_street_walkability.geojson")

gdf_crossings = gpd.read_file("../data/prep/prep_crossings_barmbek.geojson")
gdf_lsa = gpd.read_file("../data/prep/prep_lsa_barmbek.geojson")
gdf_switch = gpd.read_file("../data/prep/switch_barmbek.geojson")

# =========================
# CRS FIX (METERS!)
# =========================
TARGET_CRS = 25832  # UTM Hamburg

gdf_streets = gdf_streets.to_crs(TARGET_CRS)
gdf_crossings = gdf_crossings.to_crs(TARGET_CRS)
gdf_lsa = gdf_lsa.to_crs(TARGET_CRS)
gdf_switch = gdf_switch.to_crs(TARGET_CRS)

# =========================
# POINT TYPES
# =========================
gdf_crossings["type"] = "crossing"
gdf_lsa["type"] = "traffic_light"
gdf_switch["type"] = "mobility_hub"

# =========================
# MERGE POINTS
# =========================
gdf_points = pd.concat([
    gdf_crossings[["geometry", "type"]],
    gdf_lsa[["geometry", "type"]],
    gdf_switch[["geometry", "type"]]
], ignore_index=True)

gdf_points = gpd.GeoDataFrame(gdf_points, geometry="geometry", crs=TARGET_CRS)

# =========================
# INFRA SCORING (verbessert)
# =========================
def score_point(row):
    if row["type"] == "traffic_light":
        return 3
    elif row["type"] == "crossing":
        return 2
    elif row["type"] == "mobility_hub":
        return 4   # 🔥 stärker gewichtet
    else:
        return 1

gdf_points["infra_score"] = gdf_points.apply(score_point, axis=1)

# =========================
# STREET ID (wichtig!)
# =========================
gdf_streets = gdf_streets.reset_index().rename(columns={"index": "street_id"})

# =========================
# BUFFER (50m Einflussbereich)
# =========================
gdf_streets_buffer = gdf_streets.copy()
gdf_streets_buffer["geometry"] = gdf_streets_buffer.geometry.buffer(50)

# =========================
# SPATIAL JOIN (INTERSECTS)
# =========================
gdf_joined = gpd.sjoin(
    gdf_streets_buffer,
    gdf_points,
    how="left",
    predicate="intersects"
)

# =========================
# INFRA AGGREGATION (pro Straße)
# =========================
infra_per_street = (
    gdf_joined.groupby("street_id")["infra_score"]
    .sum()
    .reset_index()
)

# =========================
# MERGE BACK
# =========================
gdf_final = gdf_streets.merge(infra_per_street, on="street_id", how="left")

gdf_final["infra_score"] = gdf_final["infra_score"].fillna(0)

# =========================
# NORMALISIERUNG INFRA (0–5)
# =========================
gdf_final["infra_score_scaled"] = (
    gdf_final["infra_score"] / gdf_final["infra_score"].max()
) * 5

# =========================
# FINAL SCORE (0–10)
# =========================
gdf_final["final_score"] = (
    gdf_final["walkability_score"] +
    gdf_final["infra_score_scaled"]
)

# Normalisieren auf 0–10
gdf_final["final_score"] = (
    gdf_final["final_score"] - gdf_final["final_score"].min()
)

gdf_final["final_score"] = (
    gdf_final["final_score"] / gdf_final["final_score"].max()
) * 10

# =========================
# FINAL CLASS (optional)
# =========================
def classify_final(score):
    if score >= 7:
        return "High"
    elif score >= 4:
        return "Medium"
    else:
        return "Low"

gdf_final["final_class"] = gdf_final["final_score"].apply(classify_final)

# =========================
# EXPORT (WICHTIG!)
# =========================
gdf_final.to_file(
    "../data/mart/mart_walkability_with_infra.geojson",
    driver="GeoJSON"
)

# =========================
# DEBUG CHECKS
# =========================
print("Columns:", gdf_final.columns)
print("\nWalkability:")
print(gdf_final["walkability_score"].describe())

print("\nInfra:")
print(gdf_final["infra_score"].describe())

print("\nFinal:")
print(gdf_final["final_score"].describe())

print("\nSample:")
print(gdf_final.head())

Columns: Index(['street_id', 'road_type', 'surface_type_clean', 'speed_kmh', 'lanes',
       'walkability_score', 'walkability_class', 'district', 'year',
       'geometry', 'infra_score', 'infra_score_scaled', 'final_score',
       'final_class'],
      dtype='str')

Walkability:
count    519.000000
mean       5.175337
std        1.765742
min        2.000000
25%        3.000000
50%        5.000000
75%        7.000000
max        7.000000
Name: walkability_score, dtype: float64

Infra:
count    519.000000
mean       1.630058
std        2.327016
min        0.000000
25%        0.000000
50%        0.000000
75%        3.000000
max       15.000000
Name: infra_score, dtype: float64

Final:
count    519.000000
mean       4.131878
std        1.956730
min        0.000000
25%        2.222222
50%        4.074074
75%        5.555556
max       10.000000
Name: final_score, dtype: float64

Sample:
   street_id                                          road_type  \
0          0  Stadtstraße (alle Straße

In [46]:
print(gdf_final.geometry.geom_type.value_counts())

LineString         515
MultiLineString      4
Name: count, dtype: int64


In [47]:
pct = (
    streets_mart["walkability_class"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print(pct)

walkability_class
High walkability      61.7
Medium walkability    35.3
Low walkability        3.1
Name: proportion, dtype: float64


In [48]:
print(gdf_joined["infra_score"].describe())
print(gdf_joined["infra_score"].value_counts())

count    284.000000
mean       2.978873
std        0.634333
min        2.000000
25%        3.000000
50%        3.000000
75%        3.000000
max        4.000000
Name: infra_score, dtype: float64
infra_score
3.0    170
2.0     60
4.0     54
Name: count, dtype: int64


In [50]:
print(gdf_final.groupby("final_class")["final_score"].describe())

             count      mean       std       min       25%       50%  \
final_class                                                            
High          33.0  7.497194  0.674277  7.037037  7.037037  7.037037   
Low          257.0  2.392276  0.974375  0.000000  2.222222  2.222222   
Medium       229.0  5.599224  0.596709  4.074074  5.555556  5.555556   

                  75%        max  
final_class                       
High         7.777778  10.000000  
Low          3.333333   3.703704  
Medium       5.555556   6.666667  
